<a href="https://colab.research.google.com/github/juancarloszuletacorcho-ops/Juan.Zuleta/blob/main/Asig7_Practica2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instala las bibliotecas necesarias si no las tienes instaladas
!pip install pandas scikit-learn nltk spacy
!pip install -U sentence-transformers

# Caso Práctico

En este caso práctico aplicaremos un pipeline de NLP para analizar el sentimiento de los tweets. Para ello, aplicaremos los conceptos dados en las clases 1 y 2.


Descripción del Caso Práctico
El objetivo del caso práctico es:

Preprocesamiento de Texto: Limpiar y normalizar el texto utilizando técnicas como tokenización, eliminación de stopwords, lematización y normalización.
Modelado: Utilizar el algoritmo de clasificación para clasificar las reseñas como positivas o negativas.
Evaluación: Medir el desempeño del modelo con métricas cuantitativas y analizar los resultados para obtener una evaluación cualitativa.

## Parte 1: Pruebas y Visualización de represetnaciones de texto

En esta parte del caso práctico, nos enfocamos en la representación en Python. El objetivo es visualizar las representaciones de texto generadas por diferentes técnicas, como BoW, TF-IDF, Word2Vec y vectores contextuales, para comprender mejor las relaciones semánticas entre las palabras y los documentos.

__Preprocesamiento de Datos__
- Cargamos el conjunto de datos IMDB que contiene reseñas de películas y sus respectivas etiquetas de sentimiento (positivo o negativo).
- Preparamos los datos para su uso en las técnicas de representación de texto.
__Representación de Texto con TF-IDF__
- Utilizamos el esquema TF-IDF para convertir las reseñas de texto en vectores numéricos.
- Aplicamos PCA (Análisis de Componentes Principales) para reducir la dimensionalidad de los vectores TF-IDF a dos dimensiones para su visualización.
- Creamos un gráfico de dispersión utilizando Plotly para visualizar las representaciones TF-IDF en un espacio bidimensional, coloreando las reseñas según su etiqueta de sentimiento.

__Representación de Texto con Word2Vec__
- Entrenamos un modelo Word2Vec sobre las reseñas de películas para generar vectores de palabras (embeddings) que capturan las relaciones semánticas entre las palabras.
- Utilizamos PCA para reducir la dimensionalidad de los embeddings Word2Vec a dos dimensiones para su visualización.
- Creamos otro gráfico de dispersión con Plotly para visualizar las representaciones de palabras Word2Vec en un espacio bidimensional, resaltando las palabras más relevantes.

__Representación de Texto con Vecotres Contextuales__
- Empleamos un modelo pre-entrenado empleando la librería _Sentence transformers_.
- Utilizamos PCA para reducir la dimensionalidad de los embeddings a dos dimensiones para su visualización.
- Creamos otro gráfico de dispersión con Plotly para visualizar las representaciones de palabras en un espacio bidimensional, resaltando las palabras más relevantes.

__Resultados y Observaciones__
Observamos que las técnicas de representación de texto como TF-IDF y Word2Vec capturan diferentes aspectos de la información contenida en las reseñas de películas.


La visualización de embeddings nos permite explorar las relaciones semánticas entre las palabras y los documentos de una manera intuitiva.
Podemos identificar agrupaciones de palabras similares y analizar cómo se distribuyen las reseñas en el espacio de representación.

### Importación de librerías

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
import plotly.express as px

### Lectura de datos

Cargamos el conjunto de datos _IMDB_Dataset.csv_ que consiste en un recurso popular en tareas de NLP para clasificación de texto. Este conjunto de datos contiene 50.000 reseñas de diferentes películas extraídas de la de datos IMBD, divididas en partes iguales en dos categorías: reseñas positivas y reseñas negativas


In [ ]:
df = pd.read_csv('IMDB_Dataset.csv')
df = df.iloc[0:5000]
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
texts = df['review'].tolist()

### Bolsa de Palabras

Calculamos las features empleando la técnica de Bolsa de Palabras.


In [ ]:
# Bolsa de Palabras
vectorizer = CountVectorizer(max_features=1000)
X_bow = vectorizer.fit_transform(texts)
print(vectorizer.get_feature_names_out()[0:20])

['10' '15' '20' '80' 'able' 'about' 'above' 'absolutely' 'across' 'act'
 'acted' 'acting' 'action' 'actor' 'actors' 'actress' 'actual' 'actually'
 'add' 'admit']


### TF-IDF

Transformamos las reseñas en vectores de características empleando el método TF-IDF, limitando el número de características a 500.Luego, aplicamos PCA (análisis de componentes principales) para reducir la dimensionalidad de estos vectores a 2 dimensiones, lo que facilita la visualización. Finalmente, creamos un nuevo DataFrame que contiene las componentes principales y las etiquetas de clase.

En el gráfico no podemos observar ningún tipo de patrón.

In [ ]:
# Preprocesamiento de texto
vectorizer = TfidfVectorizer(max_features=500)
X_tfidf = vectorizer.fit_transform(df['review']).toarray()

# Reducción de dimensionalidad con PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_tfidf)

# Crear un DataFrame con los resultados de PCA
df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca['label'] = df['label']
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_tfidf)

# Crear un DataFrame con los resultados de PCA
df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca['label'] = df['label']
fig = px.scatter(df_pca, x='PC1', y='PC2', color=df_pca['label'].map({1: 'Positive', 0: 'Negative'}),
                 title='TF-IDF PCA Embeddings')
fig.show()

### Word2Vec

En esta celda empleamos un modelo de Word2Vec pre-entrenado. El modelo que escogemos genera vectores de 100 dimensiones.
Extraemos todos los embeddings para cada una de las reseñas y buscamos las palabras que presentan un vector de embedding más parecido a la palabra _good_.

La celda imprime por pantalla las palabras más similares junto con la similitud coseno.



In [ ]:
sentences = [text.split() for text in texts]
word2vec_model = Word2Vec(sentences, vector_size=100, window=5, min_count=5, workers=4)
print(word2vec_model.wv.most_similar('good'))

[('bad', 0.8359578847885132), ('funny', 0.8217147588729858), ('nice', 0.7969892621040344), ('great', 0.7930481433868408), ('decent', 0.7860798835754395), ('good,', 0.7496082782745361), ('budget', 0.7462834715843201), ('cool', 0.7309473752975464), ('quite', 0.7221072316169739), ('fun', 0.7196276187896729)]


Podemos entrenar el modelo Word2Vec con las reseñas que tenemos en nuestro conjunto de datos.

Posteriormente, buscamos las palabras más parecidas a la palabra _good_ según nuestro nuevo modelo. Podemos observar que las palabras ya no presentan el mismo orden ni tampoco la misma similitud coseno que antes.

In [ ]:
sentences = [text.split() for text in texts]
word2vec_model = Word2Vec(sentences, vector_size=100, window=5, min_count=5, workers=4)
word2vec_model.train(sentences, total_examples=len(sentences), epochs=5)
print(word2vec_model.wv.most_similar('good'))

[('great', 0.7315993905067444), ('bad', 0.7267882823944092), ('decent', 0.6862723231315613), ('nice', 0.6736668348312378), ('funny', 0.6226071119308472), ('cool', 0.6087554097175598), ('poor', 0.573241114616394), ('good,', 0.5506899356842041), ('stupid', 0.5488556027412415), ('fun', 0.5333776473999023)]


Podemos escoger un conjunto de palabras, calcular sus vectores y aplicar PCA con el objetivo de reducir la dimensión de 100 elementos a 2. De esta forma somos capaces posteriormente de representar los vectores de palabras (que presentan 2 dimensiones) en una gráfica.

Observamos que las palabras film, movie y story se encuentran cerca. Esto es debido a que deben aparecer en contextos similares en nuestro conjunto de datos, lo cual parece tener sentido.

También observamos que las palabras _love_ o _hate_ se encuentran cerca también. Lo mismo ocurre con las palabras _good_ y _bad_.

In [ ]:
words = ['good', 'bad', 'movie', 'film', 'story', 'great', 'awful', 'love', 'hate', 'boring']
embeddings = [word2vec_model.wv[word] for word in words]

pca = PCA(n_components=2)
result = pca.fit_transform(embeddings)

df_embeddings = pd.DataFrame(result, columns=['PCA1', 'PCA2'])
df_embeddings['word'] = words
df_embeddings['label'] = df['label']

fig = px.scatter(df_embeddings, x='PCA1', y='PCA2',
                 text='word',
                 color=df_embeddings['label'].map({1: 'Positive', 0: 'Negative'}))
fig.update_traces(textposition='top center')
fig.update_layout(title='Visualización de Word Embeddings con PCA',
                  xaxis_title='PCA1',
                  yaxis_title='PCA2')
fig.show()

### Word Embeddings Contextuales

En esta sección aprenderemos a cargar modelos contextuales pre-entrenados para poder ser usados directamente. para ello, emplearemos la librería sentence transformer: (https://sbert.net/index.html)

Esta librería permite usar muchos de los mejores modelos de embeddings contextuales de código abierto que existen. También permite re-entrenarlos por si se requiere adaptarlos para una lengua en concreto o para un dominio específico.

El modelo que emplearemos se conoce como _all-MiniLM-L6-v2_ ya que es uno de los modelos más pequeños que mejor rendimiento presenta. Los modelos disponibles se pueden consutar en https://sbert.net/docs/pretrained_models.html



La idea es la misma que para los word embeddings estáticos. Cargamos el modelo y calculamos los vectores de embedding para un conjunto de palabras de interés.

Posteriormente, aplicamos PCA para reducir las dimensiones de los embeddings que en este caso son de 768 elementos.

Por último, representamos los vectores de embeddings de 2 dimensiones para observar las relaciones entre las palabras.

Observamos que ahora las palabras que pueden considerarse antónimas como _good_ o _bad_ ya no están cerca. Además, observamos que las palabras "positivas" están juntas y las palabras "negativas" también forman un grupo.

In [ ]:
from sentence_transformers import SentenceTransformer

contextual_model = SentenceTransformer("all-MiniLM-L6-v2")

words = ['good', 'bad', 'movie', 'film', 'story', 'great', 'awful', 'love', 'hate', 'boring', "actor"]
embeddings = [contextual_model.encode(word) for word in words]

pca = PCA(n_components=2)
result = pca.fit_transform(embeddings)

df_embeddings = pd.DataFrame(result, columns=['PCA1', 'PCA2'])
df_embeddings['word'] = words
df_embeddings['label'] = df['label']

# Visualización con Plotly
fig = px.scatter(df_embeddings, x='PCA1', y='PCA2', text='word')
fig.update_traces(textposition='top center')
fig.update_layout(title='Visualización de Word Embeddings Contextuales con PCA',
                  xaxis_title='PCA1',
                  yaxis_title='PCA2')
fig.show()

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning:

`resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.



## Parte 2: Entrenamiento de modelos secuenciales
En esta parte del caso práctico, nos centramos en implementar un modelo de clasificación de reseñas de películas utilizando una red LSTM en PyTorch. El objetivo es entrenar un modelo que pueda predecir si una reseña de una película es positiva o negativa basándose en el texto de la reseña.

__Preprocesamiento de Datos__
- Dividimos el conjunto de datos IMDB en conjuntos de entrenamiento y prueba.
- Preparamos las reseñas para utilizarlas como entrada al modelo Doc2Vec, una técnica para obtener embeddings a nivel de frase.
- Entrenamos un modelo Doc2Vec sobre las reseñas de entrenamiento para generar embeddings a nivel de frase para las reseñas de entrenamiento y prueba.

__Definición del Modelo RNN__
- Definimos la arquitectura del modelo RNN utilizando la clase nn.Module de PyTorch.
- El modelo consta de una capa LSTM (Long Short-Term Memory) seguida de una capa completamente conectada (fully connected).
- Se incluye una capa de dropout para regularización y prevenir el sobreajuste.

__Entrenamiento y Evaluación del Modelo__
- Definimos las funciones train y evaluate para realizar el ciclo de entrenamiento y evaluación del modelo.
- Utilizamos el optimizador Adam y la función de pérdida BCEWithLogitsLoss.
- Entrenamos el modelo en el conjunto de entrenamiento y evaluamos su rendimiento en el conjunto de prueba durante varias épocas.
- Guardamos el mejor modelo basado en la pérdida en el conjunto de validación.

__Resultados__
- El modelo entrenado es capaz de predecir si una reseña es positiva o negativa con una cierta precisión.
- La pérdida y la precisión en el conjunto de prueba nos dan una idea de la capacidad del modelo para generalizar a datos no vistos.


### Importación de las librerías

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

### Lectura de datos

In [ ]:
# Cargar datos
df = pd.read_csv('IMDB_Dataset.csv')
df = df.iloc[0:5000]
df['label'] = df['sentiment'].apply(lambda x: 1 if x == 'positive' else 0)

### Segmentación de datos en conjunto de entrenamiento y validación

In [ ]:
# Dividir los datos en entrenamiento y prueba
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

# Preprocesamiento de texto para Doc2Vec
train_documents = [TaggedDocument(words=row.split(), tags=[i]) for i, row in enumerate(train_data['review'])]
test_documents = [row.split() for row in test_data['review']]

### Entrenamiento de modelo Doc2Vec
En esta celda entrenamos un modelo Doc2Vec para generar los embeddings a nivel de documento, es decir, a nivel de reseña. El modelo se entrena con el conjunto de datos durante 5 épocas. Los embeddings presentarán 100 dimensiones.

In [ ]:
# Entrenar el modelo Doc2Vec
doc2vec_model = Doc2Vec(vector_size=100, window=5, min_count=2, workers=4, epochs=20)
doc2vec_model.build_vocab(train_documents)
doc2vec_model.train(train_documents, total_examples=doc2vec_model.corpus_count, epochs=doc2vec_model.epochs)

Epoch: 1
	Train Loss: 0.564 | Train Acc: 72.67%
	 Val. Loss: 0.486 |  Val. Acc: 74.96%
Epoch: 2
	Train Loss: 0.467 | Train Acc: 77.65%
	 Val. Loss: 0.479 |  Val. Acc: 76.52%
Epoch: 3
	Train Loss: 0.457 | Train Acc: 78.47%
	 Val. Loss: 0.479 |  Val. Acc: 76.56%
Epoch: 4
	Train Loss: 0.448 | Train Acc: 78.70%
	 Val. Loss: 0.480 |  Val. Acc: 76.52%
Epoch: 5
	Train Loss: 0.441 | Train Acc: 79.07%
	 Val. Loss: 0.483 |  Val. Acc: 76.23%
Test Loss: 0.479 | Test Acc: 76.56%


### Cálculo de vectores
Calculamos los vectores para cada una de las reseñas y los convertimos en tensores para que posteriormente nuestro modelo de Pytorch sea capaz de procesar los vectores.

In [ ]:
# Convertir las reseñas a vectores usando Doc2Vec
X_train = np.array([doc2vec_model.infer_vector(doc.words) for doc in train_documents])
X_test = np.array([doc2vec_model.infer_vector(doc) for doc in test_documents])

# Convertir las etiquetas en tensores
y_train = torch.tensor(train_data['label'].values)
y_test = torch.tensor(test_data['label'].values)

# Convertir los datos en tensores
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)


### Definición del modelo

El modelo consiste en una red LSTM muy simple que recibirá los embeddings de Doc2vec

In [ ]:
# Definir el modelo RNN
class RNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super(RNN, self).__init__()
        self.rnn = nn.LSTM(input_dim, hidden_dim, num_layers=n_layers, bidirectional=bidirectional, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x.unsqueeze(1)  # Añadir dimensión para batch_first
        packed_output, (hidden, cell) = self.rnn(x)
        hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        return self.fc(hidden)

# Inicializar el modelo
INPUT_DIM = X_train.shape[1]
HIDDEN_DIM = 256
OUTPUT_DIM = 1
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.5

model = RNN(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, BIDIRECTIONAL, DROPOUT)


### Entrenamiento del modelo

En esta celda se definen todos los elementos necesarios para llevar a cabo el entrenamiento del modelo.

Se definen el optimizador y el criterio para la función de pérdidas.

También se define el _batch generator_ que nos permite iterar por nuestro conjunto de datos seleccionando un conjunto de N (_batch_size_) reseñas.

Por último, se calcula el accuracy del modelo tanto para el conjunto de entrenamiento como para el conjunto de evaluación con el objetivo de confirmar que el modelo es capaz de generalizar.

In [ ]:
# Definir el optimizador y la función de pérdida
optimizer = optim.Adam(model.parameters())
criterion = nn.BCEWithLogitsLoss()

# Mover el modelo y la función de pérdida a la GPU si está disponible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
criterion = criterion.to(device)

# Función de entrenamiento
def train(model, iterator, optimizer, criterion):
    epoch_loss = 0
    epoch_acc = 0
    model.train()
    for batch in iterator:
        optimizer.zero_grad()
        text, labels = batch
        text, labels = text.to(device), labels.to(device)
        predictions = model(text).squeeze(1)
        loss = criterion(predictions, labels.float())
        rounded_preds = torch.round(torch.sigmoid(predictions))
        correct = (rounded_preds == labels).float()
        acc = correct.sum() / len(correct)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_acc += acc.item()
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

# Función de evaluación
def evaluate(model, iterator, criterion):
    epoch_loss = 0
    epoch_acc = 0
    model.eval()
    with torch.no_grad():
        for batch in iterator:
            text, labels = batch
            text, labels = text.to(device), labels.to(device)
            predictions = model(text).squeeze(1)
            loss = criterion(predictions, labels.float())
            rounded_preds = torch.round(torch.sigmoid(predictions))
            correct = (rounded_preds == labels).float()
            acc = correct.sum() / len(correct)
            epoch_loss += loss.item()
            epoch_acc += acc.item()
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

# Crear iteradores manualmente
def batch_generator(X, y, batch_size=64):
    dataset = [(X[i], y[i]) for i in range(len(X))]
    for start in range(0, len(dataset), batch_size):
        end = start + batch_size
        batch = dataset[start:end]
        X_batch = torch.stack([item[0] for item in batch])
        y_batch = torch.stack([item[1] for item in batch])
        yield X_batch, y_batch

# Entrenamiento del modelo
N_EPOCHS = 5
BATCH_SIZE = 64
best_valid_loss = float('inf')

for epoch in range(N_EPOCHS):
    train_iterator = list(batch_generator(X_train, y_train, BATCH_SIZE))
    test_iterator = list(batch_generator(X_test, y_test, BATCH_SIZE))

    train_loss, train_acc = train(model, train_iterator, optimizer, criterion)
    valid_loss, valid_acc = evaluate(model, test_iterator, criterion)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'best-model.pt')

    print(f'Epoch: {epoch+1}')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

# Cargar el mejor modelo y evaluar en el conjunto de prueba
model.load_state_dict(torch.load('best-model.pt'))
test_iterator = list(batch_generator(X_test, y_test, BATCH_SIZE))
test_loss, test_acc = evaluate(model, test_iterator, criterion)
print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')


Epoch: 1
	Train Loss: 0.453 | Train Acc: 78.22%
	 Val. Loss: 0.479 |  Val. Acc: 75.74%
Epoch: 2
	Train Loss: 0.440 | Train Acc: 78.97%
	 Val. Loss: 0.482 |  Val. Acc: 76.39%
Epoch: 3
	Train Loss: 0.432 | Train Acc: 79.32%
	 Val. Loss: 0.486 |  Val. Acc: 75.94%
Epoch: 4
	Train Loss: 0.425 | Train Acc: 79.79%
	 Val. Loss: 0.489 |  Val. Acc: 75.80%
Epoch: 5
	Train Loss: 0.419 | Train Acc: 79.86%
	 Val. Loss: 0.490 |  Val. Acc: 76.09%
Test Loss: 0.479 | Test Acc: 75.74%
